# Session 3 — Data Cleaning and Preprocessing

**Data Science & Machine Learning: from Skills to the Project**  
**Duration:** 90 minutes  
**Case study:** assess and prepare the book catalogue collected in Sessions 1–2.

This notebook loads earlier catalogue exports when available. Otherwise, it creates a deterministic catalogue with realistic quality issues. The raw inputs are never overwritten.

## Outcomes and agenda

By the end, you can assess missing values, duplicates, inconsistent types and outliers; choose and explain cleaning and imputation decisions; compare common numerical scalers; and avoid leakage by fitting learned transformations on training data only.

| Minutes | Activity |
|---:|---|
| 0–10 | Inspect the catalogue |
| 10–20 | Compact quality audit |
| 20–35 | Clean types, text and invalid values |
| 35–50 | Missing values and imputation choices |
| 50–60 | Outlier decisions |
| 60–70 | Scaling and leakage |
| 70–90 | Guided cleaning function and independent challenge |

Run cells from top to bottom. The notebook uses pandas, NumPy and scikit-learn; no network access is required.

## 1. Load the catalogue, or generate a deterministic fallback

Use `data/raw/catalogue_raw.csv` and `data/raw/catalogue_scraped.csv` from the earlier sessions when they exist. The exports have overlapping but not identical schemas, so the loader maps both into common catalogue fields. If neither file is available, the fallback introduces whitespace and case variation, missing values, duplicate IDs, mixed price formats, a negative price and an unusually high price.

Keep a copy of the loaded frame as the raw input. Do not clean in place or write cleaned values back to the source files.

In [ ]:
from pathlib import Path
import re

import numpy as np
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler, RobustScaler, StandardScaler

RAW_DIR = Path("data") / "raw"
PROCESSED_DIR = Path("data") / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
RAW_PATHS = [RAW_DIR / "catalogue_raw.csv", RAW_DIR / "catalogue_scraped.csv"]
STANDARD_COLUMNS = ["record_id", "title", "author", "category", "price_gbp", "rating", "available"]

def make_fallback_catalogue():
    """Create deterministic sample records with realistic, deliberate quality issues."""
    rows = [
        ["B001", " The Hobbit ", "J. R. R. Tolkien", "fantasy", "£8.99", "4.5", "True"],
        ["B002", "Pride and Prejudice", "Jane Austen", "Fiction", "7.50", "4", "yes"],
        ["B003", "A Brief History of Time", "Stephen Hawking", "science", "9.25", None, "False"],
        ["B004", "The Left Hand of Darkness", "Ursula K. Le Guin", "Science fiction", "£ 8.25", "4.7", "true"],
        ["B005", "Dune", "Frank Herbert", "science fiction", "£12.00", "4.8", "TRUE"],
        ["B006", "Kindred", "Octavia E. Butler", "Fiction", "11.50", "4.3", "1"],
        ["B007", "The Dispossessed", "Ursula K. Le Guin", " Science Fiction ", "10.25", "4.6", "True"],
        ["B008", "The Name of the Rose", "Umberto Eco", "Mystery", "14.00", "4.1", "yes"],
        ["B009", "Invisible Cities", "Italo Calvino", None, "£9.75", "4.0", "True"],
        ["B010", "The Fifth Season", "N. K. Jemisin", "fantasy", "13.50", "4.7", "True"],
        ["B011", "The Martian", "Andy Weir", "Science Fiction", "16.00", "4.2", "1"],
        ["B012", "Piranesi", "Susanna Clarke", "Fantasy", None, "4.4", "True"],
        ["B013", "The Overstory", "Richard Powers", "Fiction", "15.25", "4.0", "False"],
        ["B014", "An Immense World", "Ed Yong", "Nature", "18.00", "4.6", "yes"],
        ["B015", "A Short History of Nearly Everything", "Bill Bryson", "Science", "-3.00", "4.2", "True"],
        ["B016", "The Atlas of Remote Islands", "Judith Schalansky", "Travel", "9999.00", "3.9", "True"],
        ["B017", "  The Dawn of Everything ", "David Graeber", "History", "17.50", "4.1", "True"],
        ["B018", "The Book of Why", "Judea Pearl", "Science", "£21.00", "4.0", "False"],
        ["B002", "Pride and Prejudice", "Jane Austen", "fiction", "7.50", "4.0", "yes"],
        ["B019", "   ", "Unknown", "Fiction", "8.00", "4.0", "True"],
    ]
    return pd.DataFrame(rows, columns=STANDARD_COLUMNS)

def align_catalogue_columns(frame):
    """Map an earlier-session export to the shared catalogue schema."""
    result = frame.copy()
    for column in STANDARD_COLUMNS:
        if column not in result.columns:
            result[column] = pd.NA
    return result[STANDARD_COLUMNS]

loaded_frames = []
loaded_files = []
for path in RAW_PATHS:
    if path.is_file():
        source_frame = pd.read_csv(path, dtype=str, keep_default_na=True)
        loaded_frames.append(align_catalogue_columns(source_frame))
        loaded_files.append(str(path))

if loaded_frames:
    raw_df = pd.concat(loaded_frames, ignore_index=True)
    input_description = ", ".join(loaded_files)
else:
    raw_df = make_fallback_catalogue()
    input_description = "deterministic fallback sample"

raw_df = raw_df.copy(deep=True)
print(f"Input: {input_description}")
print(f"Rows: {len(raw_df)} | columns: {len(raw_df.columns)} | raw input preserved in raw_df")
assert not raw_df.empty
assert raw_df.columns.tolist() == STANDARD_COLUMNS

## 2. Inspect shape, types, missingness and distributions

Start with inexpensive checks before changing data. `info()` reveals inferred types and non-null counts; descriptive statistics help expose suspicious ranges, but summary statistics do not prove that a value is valid. A mixed-type price column often appears as `object` and needs deliberate conversion.

In [ ]:
print("Shape:", raw_df.shape)
print("Dtypes:\n", raw_df.dtypes.to_string())
print("Missing values by column:\n", raw_df.isna().sum().to_string())
print("Duplicate record IDs:", int(raw_df["record_id"].duplicated(keep=False).sum()))
print("Descriptive statistics (numeric fields where available):\n", raw_df.describe(include="all").transpose().to_string())
assert raw_df.shape[0] > 0
assert "price_gbp" in raw_df.columns

## 3. Build a compact quality audit

A useful audit makes data problems measurable. Here, duplicate counts refer to rows beyond the first instance of an ID; missing counts are recorded per field. We also count unparseable prices and obvious non-positive prices without silently altering the raw input.

In [ ]:
def numeric_price_candidate(value):
    """Extract a numeric price candidate from a scalar, without modifying its source."""
    if pd.isna(value):
        return np.nan
    match = re.search(r"-?\d+(?:\.\d+)?", str(value).replace(",", ""))
    return float(match.group()) if match else np.nan

price_candidates = raw_df["price_gbp"].map(numeric_price_candidate)
quality_audit = pd.DataFrame({
    "metric": ["rows", "columns", "duplicate ID rows beyond first", "missing record IDs", "missing titles", "missing categories", "missing prices", "unparseable prices", "non-positive prices"],
    "value": [
        len(raw_df), raw_df.shape[1], int(raw_df["record_id"].duplicated().sum()),
        int(raw_df["record_id"].isna().sum()), int(raw_df["title"].isna().sum()),
        int(raw_df["category"].isna().sum()), int(raw_df["price_gbp"].isna().sum()),
        int((raw_df["price_gbp"].notna() & price_candidates.isna()).sum()),
        int((price_candidates <= 0).sum()),
    ],
})
print(quality_audit.to_string(index=False))
assert quality_audit.loc[quality_audit["metric"] == "rows", "value"].iloc[0] == len(raw_df)

## 4. Cleaning decisions and a reusable cleaning function

Decisions for this exercise: strip and normalise text; standardise category capitalisation; parse currency strings into numbers; treat non-positive or unparseable prices as missing; parse ratings and treat values outside 0–5 as missing; normalise common Boolean spellings; remove rows with a blank required ID or title; and keep the first duplicate ID, documenting the duplicate count. Optional missing values remain missing for now. The large price is retained until it can be investigated in context.

These are explicit, reviewable rules, not universal defaults. A real workflow should preserve provenance and record the reason for every exclusion or correction.

**Guided exercise:** use the reusable function below on a copy of the raw input, then review its counts, types and remaining missingness. The function returns a clean frame and decision counts without changing the source.

In [ ]:
def clean_text(value):
    if pd.isna(value):
        return pd.NA
    text = re.sub(r"\s+", " ", str(value)).strip()
    return text if text else pd.NA

def parse_number(value):
    if pd.isna(value):
        return np.nan
    match = re.search(r"-?\d+(?:\.\d+)?", str(value).replace(",", ""))
    return float(match.group()) if match else np.nan

def parse_available(value):
    if pd.isna(value):
        return pd.NA
    text = str(value).strip().casefold()
    if text in {"true", "yes", "1", "available", "in stock"}:
        return True
    if text in {"false", "no", "0", "unavailable", "out of stock"}:
        return False
    return pd.NA

def clean_catalogue(frame):
    """Return a clean copy plus counts explaining the applied row-level decisions."""
    missing_columns = set(STANDARD_COLUMNS).difference(frame.columns)
    if missing_columns:
        raise ValueError(f"Catalogue is missing required columns: {sorted(missing_columns)}")
    result = frame[STANDARD_COLUMNS].copy()
    for column in ["record_id", "title", "author", "category"]:
        result[column] = result[column].map(clean_text)
    result["record_id"] = result["record_id"].str.upper()
    result["category"] = result["category"].str.casefold().str.title()
    result["price_gbp"] = result["price_gbp"].map(parse_number)
    result.loc[result["price_gbp"] <= 0, "price_gbp"] = np.nan
    result["rating"] = result["rating"].map(parse_number)
    result.loc[~result["rating"].between(0, 5), "rating"] = np.nan
    result["available"] = result["available"].map(parse_available).astype("boolean")

    blank_required = result["record_id"].isna() | result["title"].isna()
    dropped_required = int(blank_required.sum())
    result = result.loc[~blank_required].copy()
    duplicate_rows = int(result["record_id"].duplicated().sum())
    result = result.drop_duplicates(subset="record_id", keep="first").copy()
    result["price_gbp"] = result["price_gbp"].astype("float64")
    result["rating"] = result["rating"].astype("float64")
    counts = {"dropped_blank_id_or_title": dropped_required, "duplicate_rows_removed": duplicate_rows}
    return result.reset_index(drop=True), counts

clean_df, cleaning_counts = clean_catalogue(raw_df)
assert clean_df["record_id"].is_unique
assert clean_df["record_id"].notna().all() and clean_df["title"].notna().all()
assert pd.api.types.is_float_dtype(clean_df["price_gbp"])
assert pd.api.types.is_float_dtype(clean_df["rating"])
assert str(clean_df["available"].dtype) == "boolean"
print("Cleaning decisions:", cleaning_counts)
print("Cleaned shape:", clean_df.shape, "| remaining missing values:", int(clean_df.isna().sum().sum()))
print(clean_df.head(6).to_string(index=False))

## 5. Missing values: compare defensible choices

Dropping rows can be reasonable when only a few non-critical records are incomplete; dropping a column loses all its information. Mean imputation is sensitive to skew and outliers; the median is more robust for prices; the mode suits categories or discrete values but can over-represent the most common group. Add an indicator when missingness may itself carry information.

The following small comparison is descriptive only. Its whole-sample mean/median are **not** used for a predictive evaluation. For modelling, split first and fit the imputer on the training partition only.

`SimpleImputer` supports mean, median, most-frequent and constant strategies. `KNNImputer` estimates missing values from similar rows, but is sensitive to scaling, neighbour choice and irrelevant features. Either method can introduce bias; fit it only on training data in an ML workflow.

In [ ]:
prices = clean_df["price_gbp"]
price_mode = prices.mode(dropna=True).iloc[0]
imputation_comparison = pd.DataFrame({
    "approach": ["drop missing price", "fill with mean", "fill with median", "fill with mode"],
    "resulting row count": [int(prices.notna().sum()), len(prices), len(prices), len(prices)],
    "illustrative fill value": [np.nan, prices.mean(), prices.median(), price_mode],
})
print(imputation_comparison.round(2).to_string(index=False))
assert prices.notna().any()
assert np.isfinite(prices.dropna()).all()

## 6. Outliers: detect, investigate, then decide

The IQR rule flags values below Q1 − 1.5×IQR or above Q3 + 1.5×IQR. A flag is not proof of an error: a high price may be a valid collector's edition. Correct a demonstrable data-entry error, retain plausible extremes, cap only with a justified policy, or remove records only when their inclusion is invalid for the stated task. Keep the original and document any change. The fallback's unusually high price is retained until there is evidence it is invalid; an audit flag is not an automatic cap or deletion.

In [ ]:
observed_prices = clean_df["price_gbp"].dropna()
q1, q3 = observed_prices.quantile([0.25, 0.75])
iqr = q3 - q1
lower_bound, upper_bound = q1 - 1.5 * iqr, q3 + 1.5 * iqr
outlier_mask = clean_df["price_gbp"].lt(lower_bound) | clean_df["price_gbp"].gt(upper_bound)
outlier_rows = clean_df.loc[outlier_mask, ["record_id", "title", "price_gbp"]]
print(f"IQR bounds: {lower_bound:.2f} to {upper_bound:.2f} GBP")
print("Flagged records (review; not automatically removed):")
print(outlier_rows.to_string(index=False) if not outlier_rows.empty else "None")
assert clean_df.loc[outlier_mask, "price_gbp"].notna().all()

## 7. Split first: prevent leakage when imputing and scaling

An imputer or scaler learns statistics from data. If it is fitted before the train/test split, the test set influences those statistics: this is **data leakage** and can make evaluation too optimistic. Split the cleaned, un-imputed features first; fit each learned transformation on training data, then use only `.transform()` on test data. The output CSV below is a descriptive cleaned catalogue and is not a model evaluation set.

**Scaler comparison:** MinMaxScaler maps training minima/maxima to a range but is sensitive to extremes; StandardScaler centres on the mean and scales by standard deviation, also sensitive to extremes; RobustScaler uses the median and IQR and is less influenced by extremes. None guarantees a normal distribution, and scaling is unnecessary for some algorithms. Fit on training data only. The test partition receives training-derived values and may fall outside the training range.

In [ ]:
feature_columns = ["price_gbp", "rating"]
X = clean_df[feature_columns].copy()
train_X, test_X = train_test_split(X, test_size=0.25, random_state=42)
print("Split sizes:", len(train_X), "training /", len(test_X), "test")
print("Missing values before imputation — train:", train_X.isna().sum().to_dict())
print("Missing values before imputation — test:", test_X.isna().sum().to_dict())
assert len(train_X) + len(test_X) == len(X)
assert train_X.index.intersection(test_X.index).empty

imputer = SimpleImputer(strategy="median", add_indicator=True)
train_array = imputer.fit_transform(train_X)
test_array = imputer.transform(test_X)
assert train_array.shape[1] == test_array.shape[1]
assert np.isfinite(train_array).all() and np.isfinite(test_array).all()
print("Imputer medians (training only):", imputer.statistics_.round(2).tolist())
print("Imputed feature matrix shapes:", train_array.shape, test_array.shape)

In [ ]:
scalers = {
    "MinMaxScaler": MinMaxScaler(),
    "StandardScaler": StandardScaler(),
    "RobustScaler": RobustScaler(),
}
scaler_comparison = []
for name, scaler in scalers.items():
    scaled_train = scaler.fit_transform(train_array)
    scaled_test = scaler.transform(test_array)
    assert scaled_train.shape == train_array.shape
    assert scaled_test.shape == test_array.shape
    scaler_comparison.append({
        "scaler": name,
        "training feature-0 min": scaled_train[:, 0].min(),
        "training feature-0 max": scaled_train[:, 0].max(),
        "training feature-0 median": np.median(scaled_train[:, 0]),
    })
scaler_comparison = pd.DataFrame(scaler_comparison)
print(scaler_comparison.round(2).to_string(index=False))
assert scalers["MinMaxScaler"].data_min_[0] == train_array[:, 0].min()
assert scalers["MinMaxScaler"].data_max_[0] == train_array[:, 0].max()

In [ ]:
guided_df, guided_counts = clean_catalogue(raw_df.copy())
assert guided_df["record_id"].is_unique
assert guided_df["record_id"].notna().all()
assert guided_df["title"].str.strip().ne("").all()
assert pd.api.types.is_numeric_dtype(guided_df["price_gbp"])
assert pd.api.types.is_numeric_dtype(guided_df["rating"])
print("Rows retained:", len(guided_df), "| decisions:", guided_counts)
print("Remaining missing by field:\n", guided_df.isna().sum().to_string())

## 10. Independent 20-minute challenge — clean an unseen sample

The sample below is intentionally small and unseen by the cleaning function. Apply your own cleaning rules: trim title and category, parse the price, flag invalid values, and decide what to do about missing prices and duplicate IDs. Do not overwrite the sample. Write a short justification for each decision, then validate unique IDs, non-empty titles and numeric prices.

Expected result: three rows remain after dropping the blank title and keeping the first `X03`; `£ 10.00` becomes `10.0`, while the missing price remains missing. A real workflow should check whether duplicate IDs indicate corrections or distinct products.

In [ ]:
challenge_raw = pd.DataFrame({
    "record_id": ["X01", "X02", "X03", "X03", "X04"],
    "title": ["  North Woods ", "The Glass Hotel", "Sea of Tranquility", "Sea of Tranquility", "   "],
    "category": [" fiction ", "Fiction", " Science Fiction ", "science fiction", None],
    "price_gbp": ["£ 10.00", "12", None, "12", "4.50"],
})
challenge_clean, challenge_counts = clean_catalogue(
    challenge_raw.assign(author=pd.NA, rating=pd.NA, available=pd.NA)[STANDARD_COLUMNS]
)
assert challenge_clean["record_id"].is_unique
assert challenge_clean["title"].notna().all() and challenge_clean["title"].str.strip().ne("").all()
assert pd.api.types.is_numeric_dtype(challenge_clean["price_gbp"])
assert challenge_counts["duplicate_rows_removed"] == 1
assert challenge_counts["dropped_blank_id_or_title"] == 1
assert challenge_clean["price_gbp"].isna().sum() == 1
print(challenge_clean[["record_id", "title", "category", "price_gbp"]].to_string(index=False))
print("Challenge decisions:", challenge_counts, "| missing prices retained:", int(challenge_clean["price_gbp"].isna().sum()))

## 11. Export the cleaned catalogue and quality summary

Export a clean, type-consistent dataset while preserving unresolved optional missingness. The concise summary records input/output rows, removed rows, remaining missingness and the applied decisions. Raw inputs are untouched. The exported catalogue is suitable for review and later preparation; any model evaluation must use a train-fitted preprocessing pipeline.

Guided exercise checks should find unique non-empty IDs/titles, numeric price/rating columns and nullable Boolean availability; optional missing values remain visible rather than receiving statistics calculated from the full dataset.

In [ ]:
CLEAN_PATH = PROCESSED_DIR / "catalogue_clean.csv"
SUMMARY_PATH = PROCESSED_DIR / "catalogue_quality_summary.csv"
clean_df.to_csv(CLEAN_PATH, index=False, encoding="utf-8", na_rep="")

summary = pd.DataFrame([
    {"metric": "input_rows", "value": len(raw_df)},
    {"metric": "output_rows", "value": len(clean_df)},
    {"metric": "rows_dropped_blank_required_fields", "value": cleaning_counts["dropped_blank_id_or_title"]},
    {"metric": "duplicate_rows_removed", "value": cleaning_counts["duplicate_rows_removed"]},
    {"metric": "remaining_missing_cells", "value": int(clean_df.isna().sum().sum())},
    {"metric": "missing_price_gbp", "value": int(clean_df["price_gbp"].isna().sum())},
    {"metric": "outlier_prices_flagged_and_retained", "value": int(outlier_mask.sum())},
])
summary.to_csv(SUMMARY_PATH, index=False, encoding="utf-8")
round_trip = pd.read_csv(CLEAN_PATH)
assert round_trip["record_id"].is_unique
assert round_trip["record_id"].notna().all() and round_trip["title"].notna().all()
assert pd.api.types.is_numeric_dtype(round_trip["price_gbp"])
assert pd.api.types.is_numeric_dtype(round_trip["rating"])
assert round_trip["price_gbp"].isna().sum() == clean_df["price_gbp"].isna().sum()
assert summary.loc[summary["metric"] == "output_rows", "value"].iloc[0] == len(clean_df)
print(f"Saved {len(round_trip)} rows to {CLEAN_PATH}")
print(f"Saved concise audit to {SUMMARY_PATH}")
print(summary.to_string(index=False))

## 12. Reflection and recap

1. Which audit finding would you investigate first, and what source evidence would help?
2. When might dropping an incomplete row be less defensible than imputing it?
3. Why is median price imputation more plausible than mean imputation for a skewed catalogue?
4. What does an IQR outlier flag tell you—and what does it not tell you?
5. Why must the train/test split happen before fitting an imputer or scaler?

**Expected points:** prioritise issues by impact and trace them to collection context; dropping can bias a small or systematically incomplete sample; the median is less affected by extreme prices; an outlier flag is a prompt to investigate, not proof of invalidity; fitting learned preprocessing before the split leaks test-set information.

**Recap:** preserve raw data; audit before changing it; make text, type, duplicate and missing-value rules explicit; investigate rather than automatically remove outliers; split before fitting imputers and scalers; save the cleaned catalogue and a concise decision summary for future sessions.